In [1]:
import sympy as sp
from sympy.physics.mechanics import *
import sympy.physics.vector as spv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
from sympy.physics.vector.printing import init_vprinting

init_vprinting(use_latex="mathjax")

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return ReferenceFrame(
        frame,
        latexs=(
            rf"\; {{}}^\mathcal {frame} \hat {x}",
            rf"\;{{}}^\mathcal {frame} \hat {y}",
            rf"\: {{}}^\mathcal {frame} \hat {z}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

# Problem Set 4

[Problem Set 4](./MIT8_01F16_pset4.pdf)

## Problem 4.1

![Problem 4.1](../figures/PS0401-Pulley-System.jpg)

A block of mass $m_1$ is at rest on an inclined plane that makes an angle $\theta$ with the
horizontal. The coefficient of static friction between the block and the incline surface
is $\mu_s$. A massless, inextensible string is attached to one end of the block, passes over a
fixed pulley, pulley 1, around a second freely suspended pulley, pulley 2, and is finally
attached to a fixed support. The left hand part of the string is parallel to the surface
of the inclined plane. The sections of the string coming off the suspended pulley are
vertical. The pulleys are massless, but a second block of mass $m_2$ is hung from the
suspended pulley. Gravity acts downward.

(a) What is $m_{2,\min}$, the minimum value of $m_2$ for which block 1 just barely slides up
the incline? Express your answers in terms of some or all of the variables $m_1$, $\theta$,
$\mu_s$ and $g$.

(b) What is $m_{2,\max}$, the maximum value of $m_2$ for which block 1 just barely slides
down the incline? Express your answers in terms of some or all of the variables
$m_1$, $\theta$, $\mu_s$ and $g$.

(c) Now assume that the block on the incline plane is sliding upward. The coefficient
of kinetic friction between the block and the incline surface is $\mu_k$. Find the
magnitude of the acceleration of the block on the inclined plane, $a_{x1}$. Express
your answers in terms of some or all of the variables $m_1$, $m_2$, $\theta$, $\mu_k$ and the
acceleration of gravity $g$.

In [3]:
m1, m2, mu_s, theta, g, mu_k = sp.symbols(
    "m1 m2 mu_s theta g mu_k", real=True, positive=True
)

![Problem 4.1 Free Body Diagrams](../figures/PS0401A-Pulley-System.jpg)

In [4]:
T, fs, mus, N1P = sp.symbols("T f_s mu_s N_1P", real=True, positive=True)

N = reference_frame("N")
P = reference_frame("P")

P.orient_axis(N, N.z, theta)

f_total_m1 = fs * (-P.x) + N1P * (P.y) + T * P.x + m1 * g * (-N.y)
f_total_m2 = 2 * T * (N.y) + m2 * g * (-N.y)

N2Leqn1 = sp.Eq(f_total_m1.to_matrix(N), sp.zeros(3, 1))
display(Math(sp.latex(N2Leqn1)))
N2Leqn2 = sp.Eq(f_total_m2.to_matrix(N), sp.zeros(3, 1))
display(Math(sp.latex(N2Leqn2)))

T_solve = sp.solve([N2Leqn2], T, dict=True)[0]
display(Math(rf"\text{{Tension in string with no movement}}: {sp.latex(T_solve[T])}"))

N1P_solve = sp.solve(N2Leqn1.subs(T, T_solve[T]), [N1P, fs], dict=True)[0]
N1P_solve = {k: sp.simplify(v) for k, v in N1P_solve.items()}
display(
    Math(
        rf"\text{{Normal force on }} m_1 \text{{ from pulley}}: {sp.latex(N1P_solve[N1P])}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# vertical stack
stacked_vert = sp.Matrix.vstack(f_total_m1.to_matrix(N), f_total_m2.to_matrix(N))
N2Leqn3 = sp.Eq(stacked_vert, sp.zeros(stacked_vert.shape[0], 1))
display(Math(sp.latex(N2Leqn3)))
# vertical stack

N2Leqn3_solve = sp.solve(N2Leqn3, [T, N1P, m2], dict=True)
N2Leqn3_solve = [{k: sp.simplify(v) for k, v in sol.items()} for sol in N2Leqn3_solve][
    0
]

display(
    Math(
        rf"\text{{Solutions for }} T, N_1P, f_s \text{{ when }} "
        rf"m_1 \text{{ is stationary}}: {sp.latex(N2Leqn3_solve)}"
    )
)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

⎧                                            2⋅fₛ              ⎫
⎨N_1P: g⋅m₁⋅cos(θ), T: fₛ + g⋅m₁⋅sin(θ), m₂: ──── + 2⋅m₁⋅sin(θ)⎬
⎩                                             g                ⎭

In [17]:
no_slipping_condition = sp.Lt(fs, mu_s * N2Leqn3_solve[N1P])

display(
    Math(
        rf"\text{{No-slip condition}}: {sp.latex(no_slipping_condition)}"
    )
)

m2_no_slipping = N2Leqn3_solve[m2]
display(
    Math(
        rf"\text{{Mass of }} m_2 \text{{ that will cause }} "
        rf"m_1 \text{{ not to slip}}: {sp.latex(m2_no_slipping)}"
    )
)

max_fs = mu_s * N2Leqn3_solve[N1P]
m2_range = sp.Lt( m2, m2_no_slipping.subs(fs, max_fs).factor() )
display(
    Math(
        rf"\text{{Range of }} m_2 \text{{ for no slipping}}:\boxed{{ {sp.latex(m2_range)}}}"
    )
)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>